# 4. Validação

Avaliação do modelo treinado sobre o split de teste. Caso não exista um run de treino com o `EXPERIMENT_ID` informado, o modelo base é avaliado.

## Preparação

In [1]:
import sys
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from src import config, datasets, eval as eval_mod

## Configuração

In [2]:
EXPERIMENT_ID        = "yolo-direct-all-datasets"  # identifica os pesos do treino
N_SAMPLE_PREDICTIONS = 10
DATASET_STAGE        = "interim"   # "interim" or "processed"
WEIGHTS              = ""  # deixar vazio para carregar automaticamente do run de treino

# datasets de avaliação — independentes dos usados no treino
EVAL_DATASETS = datasets.available(stage=DATASET_STAGE)
# opções: ["inside-bus-view", "passenger-deakin", "passenger-detection-bus", "crowdhuman"]

## Datasets

In [3]:
data_spec = datasets.prepare(EVAL_DATASETS, stage=DATASET_STAGE)

for name in EVAL_DATASETS:
    for split in ["valid", "test"]:
        img_dir = config.DATA_DIR / DATASET_STAGE / name / split / "images"
        lbl_dir = config.DATA_DIR / DATASET_STAGE / name / split / "labels"
        if img_dir.exists():
            n_img = len(list(img_dir.glob("*.*")))
            n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
            print(f"  {name}/{split}: {n_img} imagens | {n_lbl} labels")

  inside-bus-view/valid: 13 imagens | 13 labels
  inside-bus-view/test: 15 imagens | 15 labels
  passenger-deakin/valid: 96 imagens | 96 labels
  passenger-deakin/test: 97 imagens | 97 labels
  passenger-detection-bus/valid: 34 imagens | 34 labels
  passenger-detection-bus/test: 17 imagens | 17 labels


## Pesos

In [4]:
run_dir, weights_path = eval_mod.find_run(EXPERIMENT_ID)
weights_path = WEIGHTS or weights_path

print("Run dir:", run_dir)
print("Pesos:  ", weights_path)

Run dir: C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\yolo-direct-all-datasets_20260615_211202
Pesos:   C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\yolo-direct-all-datasets_20260615_211202\train\weights\best.pt


## Modelo

In [5]:
from ultralytics import YOLO
model = YOLO(str(weights_path))

## Avaliação

In [6]:
metrics = eval_mod.run(EXPERIMENT_ID, model, data_spec,
                       n_samples=N_SAMPLE_PREDICTIONS, run_dir=run_dir)
metrics

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lospe\_netrc.
wandb: Currently logged in as: cristianmaza (IA901-2026S1-bus-passenger-count) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb:    run 'yolo-direct-all-datasets_20260615_211202_eval' -> https://wandb.ai/IA901-2026S1-bus-passenger-count/bus-passenger-count/runs/cq59qvpv
Ultralytics 8.4.50  Python-3.12.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24576MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 7.56.0 MB/s, size: 39.3 KB)
val: Scanning C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\interim\inside-bus-view\test\labels... 129 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 129/129 1.5Kit/s 0.1s
val: New cache created: C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\interim\inside-bus-view\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.8it/s 2.4s0.3ss
                   all        129       1255      0.734      0.662      0.723      0.349
Speed: 1.2

final/test/fitness,▁
final/test/metrics/mAP50(B),▁
final/test/metrics/mAP50-95(B),▁
final/test/metrics/precision(B),▁
final/test/metrics/recall(B),▁
test/fitness,▁
test/metrics/mAP50(B),▁
test/metrics/mAP50-95(B),▁
test/metrics/precision(B),▁
test/metrics/recall(B),▁
final/test/fitness,0.34924


wandb:    run finished


{'test/metrics/precision(B)': 0.7338507668849616,
 'test/metrics/recall(B)': 0.6621513944223107,
 'test/metrics/mAP50(B)': 0.7226733591605864,
 'test/metrics/mAP50-95(B)': 0.34924066098408557,
 'test/fitness': 0.34924066098408557}